# SPECT baseline reproduction (modularised)

This notebook reproduces the baseline pipeline using the modular Python code under `src/spect/baseline/`:

**phantom/uMap → acquisition model → forward projection (clean sinogram) → alpha-scaled Poisson noise → OSEM reconstruction (clean + noisy) → save outputs**

> On Myriad, run with `PYTHONPATH=$PWD/src:$PYTHONPATH` so `import spect...` works.

## 1) Paths & output folders

We write all generated artefacts (npz + figures) into a single run folder under `outputs/` to keep results organised and easy to reference in meetings/logs.

In [1]:
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve()
OUT_DIR = REPO_ROOT / "outputs" / "baseline_v1"
FIG_DIR = OUT_DIR / "figs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)

REPO_ROOT: /Users/liuxingyu/Desktop/SPECT_DL_denoise/notebooks
OUT_DIR: /Users/liuxingyu/Desktop/SPECT_DL_denoise/notebooks/outputs/baseline_v1


## 2) Import modular baseline functions

These imports are the “integration check” that the repository structure and `PYTHONPATH` are correct.
If this fails on Myriad, it usually means `PYTHONPATH` was not exported in the batch script.

In [2]:
# export PYTHONPATH=$PWD/src:$PYTHONPATH
from spect.baseline.phantom_umap import load_template_sinogram, make_phantom_and_umap
from spect.baseline.acquisition_model import build_ubmatrix_acq_model, forward_project
from spect.baseline.noise import make_noisy_sinos, sinogram_stats  # 你 noise.py 里应该有这些
from spect.baseline.recon_osem import ReconConfig, osem_reconstruct, image_stats

print("Imports OK")

ModuleNotFoundError: No module named 'spect'

## 3) Baseline configuration

Key parameters aligned with the current baseline:

- **zooms**: keep consistent grid (SPECTUBMatrix z-sampling sensitive)
- **mu**: uniform attenuation coefficient (WeiMiao-style)
- **alphas**: count-level scaling (alpha ↑ → higher counts → cleaner)
- **OSEM settings**: subsets/subiterations

In [ ]:
# --- config (match your baseline choices) ---
zooms = (0.5, 1.0, 1.0)    
mu = 0.12                  # WeiMiao style
use_cyl_fov = True

resol_slope = 0.1
resol_sigma0 = 0.1
full_3D = False

alphas = (5.0, 1.0, 0.5, 0.05)
seed = 0

recon_cfg = ReconConfig(num_subsets=21, num_subiters=42, init_value=1.0)

print("Config:")
print("zooms", zooms, "mu", mu, "alphas", alphas, "seed", seed)
print("recon_cfg", recon_cfg)

## 4) Template sinogram → phantom & uMap (activity + attenuation)

We use the template sinogram as the geometry/metadata reference, then build:
- **activity phantom** on the common grid
- **uMap** (uniform attenuation) on the same grid

In [ ]:
templ = load_template_sinogram()
bundle = make_phantom_and_umap(templ, zooms=zooms, mu=mu, use_cyl_fov=use_cyl_fov)

print("templ:", templ.as_array().shape)
print("activity:", bundle.activity.as_array().shape)
print("umap:", bundle.umap.as_array().shape)

## 5) Build acquisition model + forward projection → clean sinogram

This is the “physics” part of the pipeline:
- Build SPECTUBMatrix acquisition model (attenuation + PSF/resolution model)
- Forward project the activity phantom to get a **clean sinogram** (projection domain)

In [ ]:
acq = build_ubmatrix_acq_model(
    templ_sino=templ,
    umap=bundle.umap,
    resol_slope=resol_slope,
    resol_sigma0=resol_sigma0,
    full_3D=full_3D,
)

clean_sino = forward_project(acq, bundle.activity, templ)
print("clean_sino shape:", clean_sino.as_array().shape)
print("clean_sino stats:", sinogram_stats(clean_sino))

## 6) Add noise: alpha-scaled Poisson sampling → noisy sinograms

We branch from the clean sinogram:

- **Clean branch**: keep `clean_sino`
- **Noisy branch**: for each alpha, generate `y_noisy ~ Poisson(alpha * y_clean) / alpha`

Alpha controls relative noise level:
- alpha ↑ → higher counts → cleaner
- alpha ↓ → lower counts → noisier

In [ ]:
noisy_sinos = make_noisy_sinos(clean_sino, alphas=alphas, seed=seed)

for a in alphas:
    print(f"alpha={a} stats:", sinogram_stats(noisy_sinos[a]))

## 7) OSEM reconstruction for clean and noisy sinograms

We reconstruct:
- `recon_clean = OSEM(clean_sino)`
- `recon_noisy[alpha] = OSEM(noisy_sino[alpha])`

These reconstructions define the later deep-learning pairing:
- input: noisy OSEM
- target: clean OSEM

In [ ]:
recon_clean = osem_reconstruct(
    clean_sino,
    acq_model=acq.acq_model,         
    img_template=bundle.activity,
    config=recon_cfg,
    use_cyl_fov=use_cyl_fov,
)
print("recon_clean stats:", image_stats(recon_clean))

recon_noisy = {}
for a in alphas:
    recon_noisy[a] = osem_reconstruct(
        noisy_sinos[a],
        acq_model=acq.acq_model,
        img_template=bundle.activity,
        config=recon_cfg,
        use_cyl_fov=use_cyl_fov,
    )
    print(f"recon_noisy alpha={a} stats:", image_stats(recon_noisy[a]))

## 8) Save outputs for later training (NPZ)

We store key arrays in a single `.npz`:
- phantom (activity)
- uMap
- clean/noisy sinograms
- clean/noisy OSEM reconstructions
- alphas

This makes downstream training (e.g. U-Net) independent of rerunning SIRF each time.

In [ ]:
import numpy as np

phantom = bundle.activity.as_array().astype(np.float32)
umap_arr = bundle.umap.as_array().astype(np.float32)

clean_sino_arr = clean_sino.as_array().astype(np.float32)
noisy_sino_stack = np.stack([noisy_sinos[a].as_array() for a in alphas]).astype(np.float32)

recon_clean_arr = recon_clean.as_array().astype(np.float32)
recon_noisy_stack = np.stack([recon_noisy[a].as_array() for a in alphas]).astype(np.float32)

npz_path = OUT_DIR / "baseline_dataset.npz"
np.savez_compressed(
    npz_path,
    phantom=phantom,
    umap=umap_arr,
    clean_sinogram=clean_sino_arr,
    noisy_sinograms=noisy_sino_stack,
    recon_clean=recon_clean_arr,
    recon_noisy=recon_noisy_stack,
    alphas=np.array(alphas, dtype=np.float32),
)
print("Saved:", npz_path)
print("Keys:", np.load(npz_path).files)

## 9) Quick visual sanity check (optional)

Optional matplotlib snapshots (and saved PNGs) for:
- recon noisy
- recon clean
- residual (noisy - clean)

Useful for meetings / debugging, but not required for the pipeline to run.

In [ ]:
import matplotlib.pyplot as plt

# 2D slice
x = recon_noisy[0.05].as_array()[0]
y = recon_clean.as_array()[0]

plt.figure()
plt.title("Recon noisy (alpha=0.05)")
plt.imshow(x)
plt.colorbar()
plt.savefig(FIG_DIR / "recon_noisy_a005.png", dpi=150, bbox_inches="tight")

plt.figure()
plt.title("Recon clean")
plt.imshow(y)
plt.colorbar()
plt.savefig(FIG_DIR / "recon_clean.png", dpi=150, bbox_inches="tight")

plt.figure()
plt.title("Diff (noisy-clean)")
plt.imshow(x - y)
plt.colorbar()
plt.savefig(FIG_DIR / "diff_a005.png", dpi=150, bbox_inches="tight")

print("Saved figs to:", FIG_DIR)